# RATIS-Net — Entraînement Scalpel à grande échelle (Wikipedia streaming)

Ce notebook stream Wikipedia (5M+ phrases) vers le Scalpel via Hugging Face Datasets. Aucun GPU requis — CPU uniquement. Sauvegarde sur Google Drive pour reprise.

## 1. Installation des dépendances

In [ ]:
!pip install -q numpy datasets

## 2. Cloner le dépôt RATIS-Net

In [ ]:
!git clone https://github.com/evinajonathan13-max/Ratiss-experimental-IA-.git ratisnet_repo

In [ ]:
import sys; sys.path.insert(0, "/content/ratisnet_repo")

## 3. Télécharger GloVe (171 MB, une seule fois)

In [ ]:
!mkdir -p /content/data/glove!curl -L -o /content/data/glove/glove.6B.zip "https://nlp.stanford.edu/data/glove.6B.zip"import zipfile; zipfile.ZipFile("/content/data/glove/glove.6B.zip").extract("glove.6B.50d.txt", "/content/data/glove/")!rm /content/data/glove/glove.6B.zipprint("GloVe prêt")

## 4. Monter Google Drive (pour sauvegarde)

In [ ]:
from google.colab import drivedrive.mount("/content/drive")import os; os.makedirs("/content/drive/MyDrive/ratisnet", exist_ok=True)print("Drive monté")

## 5. Lancer le streaming Wikipedia vers le Scalpel

In [ ]:
import sys, time, ossys.path.insert(0, "/content/ratisnet_repo")# Patch data pathos.environ["RATISS_DATA"] = "/content/data"from ratis_net.glove_tokenizer import GloveTokenizerfrom ratis_net.scalpel import ScalpelLayerfrom ratis_net.data_loader import StreamingDataLoader, ScalpelStreamingTrainerfrom pathlib import Path# ConfigurationMAX_PHRASES = 5000000  # 5M phrases = langage completCHECKPOINT_EVERY = 10000  # checkpoint toutes les 10K phrasesDRIVE_PATH = Path("/content/drive/MyDrive/ratisnet/scalpel_wikipedia.pkl")# Préparer le tokenizer (GloVe + cache topo)tok = GloveTokenizer(dim=12, n_glove=8)# Charger le cache topo existant depuis le repofrom ratis_net.topo_cache import TopoCachetok._topo_cache = TopoCache(dim=8)tok._topo_cache.load()# Charger le Scalpel (reprise si checkpoint existe)scalpel = ScalpelLayer(tok, eta=0.1, coherence_threshold=0.3)if DRIVE_PATH.exists():    scalpel.load(DRIVE_PATH)    print(f"Reprise: {scalpel.network_size()} neurones déjà appris")else:    print("Démarrage from scratch")# Streamerloader = StreamingDataLoader(dataset="wikipedia", config="20231101.en",                             max_phrases=MAX_PHRASES)trainer = ScalpelStreamingTrainer(scalpel, loader)# Entraînement avec sauvegarde Driveprint(f"Streaming {MAX_PHRASES} phrases Wikipedia vers le Scalpel...")print(f"Vitesse estimée: ~15-30 ph/s sur Colab CPU")print()t0 = time.time()n = 0for phrase in loader:    scalpel.process_phrase(phrase, t_step=n)    n += 1    if n % 1000 == 0:        dt = time.time() - t0        rate = n / dt if dt > 0 else 0        eta_h = (MAX_PHRASES - n) / rate / 3600 if rate > 0 else 0        print(f"  {n:>8d} | {scalpel.network_size():>8d} neurones | {rate:.0f} ph/s | ETA {eta_h:.1f}h")    if n % CHECKPOINT_EVERY == 0:        scalpel.save(DRIVE_PATH)        print(f"  >>> Checkpoint sauvegardé sur Drive ({scalpel.network_size()} neurones)")# Sauvegarde finalescalpel.save(DRIVE_PATH)dt = time.time() - t0print(f"=== TERMINÉ ===")print(f"Phrases: {n}")print(f"Neurones: {scalpel.network_size()}")print(f"Renforcements: {scalpel.total_reinforcements}")print(f"Temps: {dt/3600:.1f}h")print(f"Taille: {os.path.getsize(DRIVE_PATH)/1024/1024:.1f} MB")print(f"Sauvegardé: {DRIVE_PATH}")

## 6. Tester la reconstruction (Synchrotron)

In [ ]:
from ratis_net.ratiss_synchrotron import RatissSynchrotron# Indexer un sous-ensemble pour testcorpus = []for i, phrase in enumerate(loader):    if i >= 5000: break    corpus.append(phrase)engine = RatissSynchrotron(scalpel=scalpel, scalpel_weight=0.4)engine.build_corpus(corpus)queries = ["what is quantum mechanics", "i feel happy today", "explain gravity"]for q in queries:    r = engine.generate_response(q)    rec = r["reconstruction"]    print(f"Q: {q}")    print(f"  R: {rec["reconstructed"][:120]}")    print(f"  coh={rec["avg_coherence"]:.3f}")

## Notes

- Colab gratuit: 12h max par session. Le script sauvegarde sur Drive toutes les 10K phrases → reprise possible.

- Si timeout: relance la cellule 5, le Scalpel reprend depuis le dernier checkpoint.

- Pour Kaggle: remplace  par .

- Pour Lightning AI: remplace par .